# Generator Dispatch Explorer — Notebook 1.2

**Learning objectives:**
- Understand merit order dispatch — baseload vs. peaking operation
- Identify marginal generators (price-setting units)
- Analyze capacity factors and detect outages
- Compare regional generation mixes

**Required cache command:**
```bash
uv run import_nem_data.py --start 2025/01/01 --end 2025/12/31 --dispatchload
```

**Also required:**
- Save the AEMO registration workbook `NEM Registration and Exemption List.xlsx` in `data/nemosis_cache/`.

**Data strategy:**
DISPATCHLOAD is 90-120x larger than DISPATCHPRICE (~50M rows vs. 500K rows). Strategic Sampling should result in 75% fewer rows, 80% less memory than brute-force full-year load
1. **Use Polars** for lazy evaluation and efficient memory usage
2. **Read parquet files directly** from NEMOSIS cache (bypass `dynamic_data_compiler`)
3. **Strategic sampling**:
   - **One week** for merit order stacks and marginal generator analysis (~1M rows)
   - **12 weeks** (one per month) for capacity factor and regional mix (~12M rows)



In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta

# NEMOSIS cache location (same as Price Explorer)
CACHE = Path('../data/nemosis_cache')

# If NEMOSIS has cached DISPATCHLOAD, the parquet files are here:
# ./data/nemosis_cache/DISPATCHLOAD_<start>_<end>.parquet
# We'll read these directly instead of using dynamic_data_compiler

# Discover what DISPATCHLOAD files exist in cache
dispatchload_files = sorted(CACHE.glob('*DISPATCHLOAD*.parquet'))

print(f"Found {len(dispatchload_files)} DISPATCHLOAD parquet files in cache:\n")
for f in dispatchload_files:
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:60s}  {size_mb:6.1f} MB")

if len(dispatchload_files) == 0:
    print("\nNo cached files found. Run NEMOSIS dynamic_data_compiler first to download data.")
    print("Example:")
    print("  from nemosis import dynamic_data_compiler")
    print("  dynamic_data_compiler('2025/07/01 00:00:00', '2025/08/01 00:00:00',")
    print("                        'DISPATCHLOAD', '../data/nemosis_cache', fformat='parquet')")
else:
    total_size = sum(f.stat().st_size for f in dispatchload_files) / 1e6
    print(f"\nTotal cache size: {total_size:.1f} MB")

In [ ]:
# Load registration data from AEMO static table
# Download "NEM Registration and Exemption List" from:
#   https://aemo.com.au/energy-systems/electricity/national-electricity-market-nem/participate-in-the-market/registration
# Save to: ./data/nemosis_cache/

# Support both .xls and .xlsx formats
reg_xlsx = CACHE / 'NEM Registration and Exemption List.xlsx'
reg_xls = CACHE / 'NEM Registration and Exemption List.xls'

if reg_xlsx.exists():
    registration_pd = pd.read_excel(reg_xlsx, sheet_name='PU and Scheduled Loads', engine='openpyxl')
elif reg_xls.exists():
    registration_pd = pd.read_excel(reg_xls, sheet_name='Generators and Scheduled Loads', engine='xlrd')
else:
    raise FileNotFoundError(
        f"Registration file not found in {CACHE}\n"
        f"Download 'NEM Registration and Exemption List' from AEMO and save to cache dir."
    )

# For xlsx: use "Fuel Source - Descriptor" (Black Coal, Natural Gas, etc.)
# instead of generic "Fuel Source - Primary" (Fossil, Solar, etc.)
if 'Fuel Source - Descriptor' in registration_pd.columns:
    registration_pd['FUEL_SOURCE_PRIMARY'] = registration_pd['Fuel Source - Descriptor'].fillna(
        registration_pd['Fuel Source - Primary']
    )
else:
    registration_pd = registration_pd.rename(columns={'Fuel Source - Primary': 'FUEL_SOURCE_PRIMARY'})

registration_pd = registration_pd.rename(columns={
    'Reg Cap generation (MW)': 'REG_CAPACITY',   # xlsx format
    'Reg Cap (MW)': 'REG_CAPACITY',               # xls format (fallback)
    'Region': 'REGIONID',
    'Station Name': 'STATIONNAME',
})

registration_pd['REG_CAPACITY'] = pd.to_numeric(registration_pd['REG_CAPACITY'], errors='coerce')
registration_pd = registration_pd[['DUID', 'FUEL_SOURCE_PRIMARY', 'REG_CAPACITY', 'REGIONID', 'STATIONNAME']]

# Convert to Polars and deduplicate
registration = pl.from_pandas(registration_pd).unique(subset=['DUID'], keep='last')

print(f"Registration records: {len(registration):,}")
print(f"\nFuel type distribution:")
print(registration.group_by('FUEL_SOURCE_PRIMARY').len().sort('len', descending=True))
print("\nSample:")
print(registration.head(10))

In [ ]:
# Create simplified fuel type mapping
fuel_map = {
    # Coal
    'Brown Coal': 'Coal', 'Black Coal': 'Coal',
    'Coal Seam Methane': 'Coal', 'Waste Coal Mine Gas': 'Coal',
    # Gas / liquid fuels
    'Natural Gas': 'Gas', 'Natural Gas / Fuel Oil': 'Gas',
    'Natural Gas / Diesel': 'Gas', 'Natrual Gas/ Diesel': 'Gas',
    'Diesel': 'Gas', 'Kerosene': 'Gas', 'Ethane': 'Gas',
    # Renewables
    'Wind': 'Wind', 'Solar': 'Solar', 'Hydro': 'Hydro',
    # Battery
    'Battery': 'Battery', 'Battery Storage': 'Battery', 'Battery / Solar': 'Battery',
    # Biomass
    'Biomass': 'Biomass', 'Bagasse': 'Biomass',
    'Renewable/ Biomass / Waste': 'Biomass',
    'Renewable/ Biomass / Waste and Fossil': 'Biomass',
}

registration = registration.with_columns(
    pl.col('FUEL_SOURCE_PRIMARY')
    .replace_strict(fuel_map, default='Other')
    .alias('FUEL_TYPE')
)

print("Simplified fuel types:")
print(registration.group_by('FUEL_TYPE').len().sort('len', descending=True))

## Strategy 1: Load One Week for Merit Order Analysis

For merit order stacks and marginal generator identification, we only need a **representative week** of data. This gives us ~1M rows instead of 50M — completely manageable.

Week selected: **2025-10-13 to 2025-10-20** (mid-spring: high solar, evening gas peaks, wind overnight)

In [ ]:
# Define target week — mid-October (spring): high solar, evening gas peaks, wind overnight
start_date = '2025-10-13'
end_date = '2025-10-20'  # Exclusive upper bound

# Find parquet files that cover this date range
target_files = [
    f for f in dispatchload_files
    if '202510' in f.name  # October 2025
]

print(f"Loading dispatch data for {start_date} to {end_date}...")
print(f"Reading from: {[f.name for f in target_files]}\n")

# NEMOSIS parquet files store all columns as strings — cast to proper types
dispatch_week = (
    pl.scan_parquet(target_files)
    .with_columns([
        pl.col('SETTLEMENTDATE').str.strptime(pl.Datetime, '%Y/%m/%d %H:%M:%S'),
        pl.col('INITIALMW').cast(pl.Float64),
        pl.col('INTERVENTION').cast(pl.Int32),
    ])
    .filter(
        (pl.col('SETTLEMENTDATE') >= datetime(2025, 10, 13)) &
        (pl.col('SETTLEMENTDATE') < datetime(2025, 10, 20)) &
        (pl.col('INTERVENTION') == 0)
    )
    .select([
        'SETTLEMENTDATE',
        'DUID',
        pl.col('INITIALMW').alias('DISPATCHLOAD'),
    ])
    .collect()
)

# Align time convention: interval start, not end
dispatch_week = dispatch_week.with_columns(
    (pl.col('SETTLEMENTDATE') - pl.duration(minutes=5)).alias('SETTLEMENTDATE')
)

print(f"Loaded {len(dispatch_week):,} rows")
print(f"Memory usage: {dispatch_week.estimated_size('mb'):.1f} MB")
print(f"Unique generators: {dispatch_week['DUID'].n_unique():,}")
print(f"Date range: {dispatch_week['SETTLEMENTDATE'].min()} to {dispatch_week['SETTLEMENTDATE'].max()}")
print("\nSample:")
print(dispatch_week.head())

In [ ]:
# Join with registration to get fuel types
dispatch_week_enriched = dispatch_week.join(
    registration.select(['DUID', 'FUEL_TYPE', 'REG_CAPACITY', 'REGIONID']),
    on='DUID',
    how='left'
).filter(
    pl.col('FUEL_TYPE').is_not_null()  # Drop unmapped DUIDs
)

print(f"Enriched data: {len(dispatch_week_enriched):,} rows")
print(f"\nFuel type energy (MWh):")
fuel_energy = (dispatch_week_enriched
    .group_by('FUEL_TYPE')
    .agg((pl.col('DISPATCHLOAD').sum() / 12).alias('MWh'))  # 5-min to hourly
    .sort('MWh', descending=True)
)
print(fuel_energy)

In [ ]:
# Merit Order Stack — VIC, single day (2025-10-15, spring Wednesday)
target_date = datetime(2025, 10, 15)
region = 'VIC1'

# Convert to pandas for matplotlib (easier plotting)
day_data_pd = (dispatch_week_enriched
    .filter(
        (pl.col('SETTLEMENTDATE').dt.date() == target_date.date()) &
        (pl.col('REGIONID') == region)
    )
    .with_columns(pl.col('SETTLEMENTDATE').dt.hour().alias('hour'))
    .group_by(['hour', 'FUEL_TYPE'])
    .agg(pl.col('DISPATCHLOAD').sum())
    .to_pandas()
)

# Pivot for stacked area chart
stack = day_data_pd.pivot(index='hour', columns='FUEL_TYPE', values='DISPATCHLOAD').fillna(0)

# Clip negative values (battery/hydro charging) — stacked area requires all-positive
stack = stack.clip(lower=0)

# Define stack order and colors
stack_order = ['Coal', 'Hydro', 'Wind', 'Solar', 'Gas', 'Battery', 'Biomass', 'Other']
stack_colors = {
    'Coal': '#4A4A4A',
    'Gas': '#FF6B6B',
    'Hydro': '#4ECDC4',
    'Wind': '#95E1D3',
    'Solar': '#FFD93D',
    'Battery': '#9D65C9',
    'Biomass': '#6BCB77',
    'Other': '#B8B8B8'
}

stack_order_present = [f for f in stack_order if f in stack.columns]

fig, ax = plt.subplots(figsize=(14, 6))
stack[stack_order_present].plot.area(
    ax=ax,
    stacked=True,
    color=[stack_colors[f] for f in stack_order_present],
    alpha=0.9,
    linewidth=0
)

ax.set_xlabel('Hour of day')
ax.set_ylabel('Dispatch (MW)')
ax.set_title(f'{region} Generation Mix — {target_date.strftime("%Y-%m-%d")}')
ax.legend(title='Fuel Type', loc='upper left')
ax.set_xticks(range(24))
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Peak demand: {stack[stack_order_present].sum(axis=1).max():.0f} MW at hour {stack[stack_order_present].sum(axis=1).idxmax()}")
print(f"Min demand: {stack[stack_order_present].sum(axis=1).min():.0f} MW at hour {stack[stack_order_present].sum(axis=1).idxmin()}")

In [ ]:
# Merit Order Stack — SA (same day for comparison)
region = 'SA1'

day_data_pd = (dispatch_week_enriched
    .filter(
        (pl.col('SETTLEMENTDATE').dt.date() == target_date.date()) &
        (pl.col('REGIONID') == region)
    )
    .with_columns(pl.col('SETTLEMENTDATE').dt.hour().alias('hour'))
    .group_by(['hour', 'FUEL_TYPE'])
    .agg(pl.col('DISPATCHLOAD').sum())
    .to_pandas()
)

stack = day_data_pd.pivot(index='hour', columns='FUEL_TYPE', values='DISPATCHLOAD').fillna(0)

# Clip negative values (battery/hydro charging) — stacked area requires all-positive
stack = stack.clip(lower=0)

stack_order_present = [f for f in stack_order if f in stack.columns]

fig, ax = plt.subplots(figsize=(14, 6))
stack[stack_order_present].plot.area(
    ax=ax,
    stacked=True,
    color=[stack_colors[f] for f in stack_order_present],
    alpha=0.9,
    linewidth=0
)

ax.set_xlabel('Hour of day')
ax.set_ylabel('Dispatch (MW)')
ax.set_title(f'{region} Generation Mix — {target_date.strftime("%Y-%m-%d")}')
ax.legend(title='Fuel Type', loc='upper left')
ax.set_xticks(range(24))
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Peak demand: {stack[stack_order_present].sum(axis=1).max():.0f} MW at hour {stack[stack_order_present].sum(axis=1).idxmax()}")
print(f"Min demand: {stack[stack_order_present].sum(axis=1).min():.0f} MW at hour {stack[stack_order_present].sum(axis=1).idxmin()}")

In [ ]:
# Marginal vs Floor Generator — VIC, one day
# Marginal = highest-SRMC fuel dispatching (price setter)
# Floor    = lowest-SRMC fuel dispatching (cheapest source running)

region = 'SA1'

# SRMC ranking: higher = more expensive
srmc_rank = {
    'Wind': 1, 'Solar': 1,
    'Hydro': 2,
    'Coal': 3,
    'Biomass': 4,
    'Gas': 5,
    'Battery': 6,
}

day_marginal = (dispatch_week_enriched
    .filter(
        (pl.col('SETTLEMENTDATE').dt.date() == target_date.date()) &
        (pl.col('REGIONID') == region) &
        (pl.col('DISPATCHLOAD') > 0) &
        (pl.col('FUEL_TYPE') != 'Other')  # Exclude unclassified
    )
    .with_columns([
        pl.col('FUEL_TYPE').replace_strict(
            {k: str(v) for k, v in srmc_rank.items()}, default='0'
        ).cast(pl.Int32).alias('SRMC_RANK'),
        pl.col('SETTLEMENTDATE').dt.hour().alias('hour'),
    ])
)

# Per interval: fuel type with max rank (marginal/price-setter) and min rank (floor/cheapest)
marginal = (day_marginal
    .group_by('SETTLEMENTDATE')
    .agg([
        pl.col('FUEL_TYPE').sort_by('SRMC_RANK', descending=True).first().alias('MARGINAL_FUEL'),
        pl.col('FUEL_TYPE').sort_by('SRMC_RANK', descending=False).first().alias('FLOOR_FUEL'),
        pl.col('hour').first(),
    ])
    .to_pandas()
)

# Count by hour for each panel
marginal_counts = marginal.groupby(['hour', 'MARGINAL_FUEL']).size().unstack(fill_value=0)
floor_counts    = marginal.groupby(['hour', 'FLOOR_FUEL']).size().unstack(fill_value=0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

marginal_counts.plot.bar(
    ax=ax1, stacked=True, width=0.8,
    color=[stack_colors.get(f, '#B8B8B8') for f in marginal_counts.columns]
)
ax1.set_ylabel('5-min intervals')
ax1.set_title(f'{region} — {target_date.strftime("%Y-%m-%d")}\n'
              f'Price setter: highest-SRMC fuel dispatching each interval')
ax1.legend(title='Marginal fuel', loc='upper left')
ax1.grid(alpha=0.3, axis='y')

floor_counts.plot.bar(
    ax=ax2, stacked=True, width=0.8,
    color=[stack_colors.get(f, '#B8B8B8') for f in floor_counts.columns]
)
ax2.set_xlabel('Hour of day')
ax2.set_ylabel('5-min intervals')
ax2.set_title('Cheapest source: lowest-SRMC fuel dispatching each interval')
ax2.legend(title='Floor fuel', loc='upper left')
ax2.set_xticklabels(range(24), rotation=0)
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Price setter summary:")
print(marginal['MARGINAL_FUEL'].value_counts().to_string())
print("\nFloor fuel summary:")
print(marginal['FLOOR_FUEL'].value_counts().to_string())

## Strategy 2: Monthly Sampling for Capacity Factor Analysis

Capacity factor requires the full year of data. Instead of loading 50M rows, we:
1. Load **one week per month** (12 weeks total)
2. Calculate weekly capacity factors
3. Extrapolate to annual estimates

This gives us ~12M rows instead of 50M — a 75% reduction while still capturing seasonal variation.

In [ ]:
# Define one representative week per month (mid-month to avoid month-boundary effects)
sample_weeks = [
    ('2025-01-13', '2025-01-20'),
    ('2025-02-10', '2025-02-17'),
    ('2025-03-10', '2025-03-17'),
    ('2025-04-14', '2025-04-21'),
    ('2025-05-12', '2025-05-19'),
    ('2025-06-09', '2025-06-16'),
    ('2025-07-14', '2025-07-21'),  # Already loaded above
    ('2025-08-11', '2025-08-18'),
    ('2025-09-08', '2025-09-15'),
    ('2025-10-13', '2025-10-20'),
    ('2025-11-10', '2025-11-17'),
    ('2025-12-08', '2025-12-15'),
]

print("Loading 12 weekly samples (one per month)...\n")

# Check which months have cached parquet files
available_months = set()
for f in dispatchload_files:
    # NEMOSIS filenames: PUBLIC_ARCHIVE#DISPATCHLOAD#FILE01#202501010000.parquet
    parts = f.stem.split('#')
    if len(parts) >= 4:
        date_str = parts[3]  # e.g., '202501010000'
        ym = date_str[:6]    # e.g., '202501'
        available_months.add(ym)

print(f"Available months in cache: {sorted(available_months)}")
print(f"\nIf some months are missing, run NEMOSIS dynamic_data_compiler to download them first.\n")

In [ ]:
# Load all weekly samples (this will take a few minutes)
# Only load if you have the parquet files cached

samples = []

for start, end in sample_weeks:
    ym = start[:7].replace('-', '')  # '2025-01' → '202501'

    # Find parquet file for this month
    month_files = [f for f in dispatchload_files if ym in f.name]

    if not month_files:
        print(f"Skipping {start} to {end} — no cached parquet for {ym}")
        continue

    print(f"Loading {start} to {end}...", end=' ')

    start_dt = datetime.strptime(start, '%Y-%m-%d')
    end_dt = datetime.strptime(end, '%Y-%m-%d')

    week_data = (
        pl.scan_parquet(month_files)
        .with_columns([
            pl.col('SETTLEMENTDATE').str.strptime(pl.Datetime, '%Y/%m/%d %H:%M:%S'),
            pl.col('INITIALMW').cast(pl.Float64),
            pl.col('INTERVENTION').cast(pl.Int32),
        ])
        .filter(
            (pl.col('SETTLEMENTDATE') >= start_dt) &
            (pl.col('SETTLEMENTDATE') < end_dt) &
            (pl.col('INTERVENTION') == 0)
        )
        .select([
            (pl.col('SETTLEMENTDATE') - pl.duration(minutes=5)).alias('SETTLEMENTDATE'),
            'DUID',
            pl.col('INITIALMW').alias('DISPATCHLOAD'),
        ])
        .collect()
    )

    samples.append(week_data)
    print(f"{len(week_data):,} rows")

if samples:
    dispatch_annual_sample = pl.concat(samples)
    print(f"\nTotal sampled data: {len(dispatch_annual_sample):,} rows")
    print(f"Memory: {dispatch_annual_sample.estimated_size('mb'):.1f} MB")
else:
    print("\nNo samples loaded. Download parquet files first using NEMOSIS.")

In [ ]:
# Enrich annual sample once â reused in capacity factor, regional mix, and monthly stack cells
dispatch_annual_sample_enriched = (
    dispatch_annual_sample
    .join(
        registration.select(['DUID', 'FUEL_TYPE', 'REG_CAPACITY', 'REGIONID']),
        on='DUID',
        how='left'
    )
    .filter(pl.col('FUEL_TYPE').is_not_null())
)

print(f"Enriched annual sample: {len(dispatch_annual_sample_enriched):,} rows")

In [ ]:
# Calculate capacity factors from sampled data
# sample_intervals based on actually loaded weeks (skipped months excluded)

energy_by_duid = (dispatch_annual_sample_enriched
    .group_by('DUID')
    .agg([
        pl.col('DISPATCHLOAD').sum().alias('TOTAL_DISPATCH'),
        pl.col('REG_CAPACITY').first(),
        pl.col('FUEL_TYPE').first(),
        pl.col('REGIONID').first()
    ])
)

# Number of 5-min intervals in our sample (based on actually loaded weeks)
sample_intervals = len(samples) * 7 * 24 * 12

energy_by_duid = energy_by_duid.with_columns(
    (pl.col('TOTAL_DISPATCH') / (pl.col('REG_CAPACITY') * sample_intervals)).alias('CF_SAMPLE')
).filter(
    (pl.col('REG_CAPACITY') > 0) & (pl.col('TOTAL_DISPATCH') > 0)
)

print(f"Generators with non-zero output: {len(energy_by_duid)}")
print("\nCapacity factor by fuel type (median from sample):")
cf_by_fuel = (energy_by_duid
    .group_by('FUEL_TYPE')
    .agg(pl.col('CF_SAMPLE').median().alias('Median_CF'))
    .sort('Median_CF', descending=True)
)
print(cf_by_fuel)

# Convert to pandas for plotting
energy_pd = energy_by_duid.to_pandas()

# Scatter: CF vs. capacity
fig, ax = plt.subplots(figsize=(12, 7))

for fuel in ['Coal', 'Gas', 'Hydro', 'Wind', 'Solar', 'Battery']:
    subset = energy_pd[energy_pd['FUEL_TYPE'] == fuel]
    if len(subset) > 0:
        ax.scatter(
            subset['REG_CAPACITY'],
            subset['CF_SAMPLE'] * 100,
            label=fuel,
            color=stack_colors.get(fuel, '#B8B8B8'),
            alpha=0.6,
            s=50
        )

ax.set_xlabel('Registered Capacity (MW)')
ax.set_ylabel('Capacity Factor (%) — from 12-week sample')
ax.set_title('Generator Capacity Factor vs. Nameplate Capacity')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(0, None)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# Identify low-CF coal units (potential outages)
coal_units = energy_pd[energy_pd['FUEL_TYPE'] == 'Coal'].copy()
coal_units = coal_units.sort_values('CF_SAMPLE')

print("Coal units with capacity factor < 50%:")
low_cf = coal_units[coal_units['CF_SAMPLE'] < 0.5]
print(low_cf[['DUID', 'REG_CAPACITY', 'CF_SAMPLE', 'REGIONID']])

if len(low_cf) > 0:
    print("\nThese units may have experienced extended outages.")
else:
    print("\nAll coal units operated at normal capacity factors.")

In [ ]:
# Wind and Solar CF distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wind_cf = energy_pd[energy_pd['FUEL_TYPE'] == 'Wind']['CF_SAMPLE'] * 100
axes[0].hist(wind_cf, bins=30, color=stack_colors['Wind'], edgecolor='black', alpha=0.7)
axes[0].axvline(wind_cf.median(), color='red', linestyle='--', linewidth=2,
                label=f'Median: {wind_cf.median():.1f}%')
axes[0].set_xlabel('Capacity Factor (%)')
axes[0].set_ylabel('Number of wind farms')
axes[0].set_title('Wind Farm CF Distribution (12-week sample)')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

solar_cf = energy_pd[energy_pd['FUEL_TYPE'] == 'Solar']['CF_SAMPLE'] * 100
axes[1].hist(solar_cf, bins=30, color=stack_colors['Solar'], edgecolor='black', alpha=0.7)
axes[1].axvline(solar_cf.median(), color='red', linestyle='--', linewidth=2,
                label=f'Median: {solar_cf.median():.1f}%')
axes[1].set_xlabel('Capacity Factor (%)')
axes[1].set_ylabel('Number of solar farms')
axes[1].set_title('Solar Farm CF Distribution (12-week sample)')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Wind: {len(wind_cf)} farms, median = {wind_cf.median():.1f}%")
print(f"Solar: {len(solar_cf)} farms, median = {solar_cf.median():.1f}%")

In [ ]:
# Regional generation mix from annual sample
regional_mix = (dispatch_annual_sample_enriched
    .group_by(['REGIONID', 'FUEL_TYPE'])
    .agg(pl.col('DISPATCHLOAD').sum())
    .to_pandas()
    .pivot(index='REGIONID', columns='FUEL_TYPE', values='DISPATCHLOAD')
    .fillna(0)
)

# Convert to percentage
regional_mix_pct = regional_mix.div(regional_mix.sum(axis=1), axis=0) * 100

stack_order_present = [f for f in stack_order if f in regional_mix_pct.columns]

fig, ax = plt.subplots(figsize=(10, 6))
regional_mix_pct[stack_order_present].plot.bar(
    ax=ax,
    stacked=True,
    color=[stack_colors[f] for f in stack_order_present],
    width=0.7
)

ax.set_xlabel('Region')
ax.set_ylabel('Energy share (%)')
ax.set_title('NEM Generation Mix by Region (12-week sample)')
ax.legend(title='Fuel Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(regional_mix_pct.index, rotation=0)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

print("\nRegional generation mix (%):")
print(regional_mix_pct.round(1))

In [ ]:
# Top 20 generators by energy output (from sample)
top_20 = (
    energy_pd.nlargest(20, 'TOTAL_DISPATCH')
    .copy()
    .assign(ENERGY_GWh=lambda x: x['TOTAL_DISPATCH'] / 12 / 1000)
)

print("Top 20 generators (from 12-week sample):")
print(top_20[['DUID', 'FUEL_TYPE', 'REGIONID', 'REG_CAPACITY', 'ENERGY_GWh', 'CF_SAMPLE']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(
    range(len(top_20)),
    top_20['ENERGY_GWh'],
    color=[stack_colors.get(f, '#B8B8B8') for f in top_20['FUEL_TYPE']]
)

ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['DUID'], fontsize=9)
ax.set_xlabel('Energy output (GWh, 12-week sample)')
ax.set_title('Top 20 NEM Generators by Energy Output')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Monthly Dispatch Stack — SA1
# Shows how the generation mix in SA shifts month-to-month.
# Wind resource is strongest in winter; solar peaks in summer;
# gas fills the gaps. Each bar represents one sampled week scaled
# to approximate monthly energy, so magnitudes are relative.

region = "SA1"

monthly_stack_pd = (
    dispatch_annual_sample_enriched
    .filter(pl.col("REGIONID") == region)
    .with_columns(pl.col("SETTLEMENTDATE").dt.month().alias("month"))
    .group_by(["month", "FUEL_TYPE"])
    .agg(pl.col("DISPATCHLOAD").sum())
    .to_pandas()
    .pivot(index="month", columns="FUEL_TYPE", values="DISPATCHLOAD")
    .fillna(0)
    .sort_index()
)

# Convert 5-min MW sums to GWh
monthly_stack_gwh = monthly_stack_pd / 12 / 1000

stack_order_present = [f for f in stack_order if f in monthly_stack_gwh.columns]

month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(12, 6))
monthly_stack_gwh[stack_order_present].plot.bar(
    ax=ax,
    stacked=True,
    color=[stack_colors[f] for f in stack_order_present],
    width=0.8
)

ax.set_xlabel("Month")
ax.set_ylabel("Energy (GWh) — one sampled week per month")
ax.set_title(f"{region} Monthly Generation Stack — 2025 (12-week sample)")
ax.legend(title="Fuel Type", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.set_xticklabels(
    [month_labels[i - 1] for i in monthly_stack_gwh.index],
    rotation=0
)
plt.tight_layout()
plt.show()

print("Monthly energy by fuel type (GWh, from sampled week):")
print(monthly_stack_gwh[stack_order_present].round(1))

## Summary: Memory-Efficient Data Strategy

**What we did:**
1. Used **Polars** for lazy evaluation and efficient memory usage
2. Read **parquet files directly** from NEMOSIS cache (no dynamic_data_compiler)
3. Applied **strategic sampling**:
   - One week for merit order stacks (1M rows)
   - 12 weeks (one per month) for capacity factor analysis (12M rows)

**Memory comparison:**
- Full year brute-force: ~50M rows, ~4-6 GB RAM
- Our approach: ~12M rows, ~800 MB RAM
- **Reduction: 75% fewer rows, 80% less memory**

**Trade-offs:**
- ✅ Fits in laptop RAM, even in WSL
- ✅ All learning objectives achieved
- ✅ Seasonal variation captured via monthly samples
- ⚠️ Capacity factors are estimates (not exact annual)
- ⚠️ Miss rare events (e.g., a single-day outage in an unsampled week)

**When to use full data:**
- Price spike autopsy (Notebook 1.3) — you want every 5-min interval for that specific event
- PyPSA model calibration (Stage 2) — comparing model to reality needs exact dispatch
- Production analytics on a proper server with 16+ GB RAM

This is the reality of energy data work: you learn to balance **completeness vs. practicality**.